# A5 — signature reversal and nearest measured cell lines

Replaces the ODE panel. Every displayed viability is a measurement.
Smoke/CI uses `synthetic_smoke` or a committed table proxy — never silent full-LINCS labelling.


In [ ]:
from pathlib import Path
import sys, json, warnings
warnings.filterwarnings("ignore")

cwd = Path.cwd().resolve()
for cand in [cwd, *cwd.parents]:
    if (cand / "src" / "gate.py").is_file():
        sys.path.insert(0, str(cand / "src"))
        break
    nested = cand / "v2"
    if (nested / "src" / "gate.py").is_file():
        sys.path.insert(0, str(nested / "src"))
        break

from paths import ensure_src_on_path, resolve_v2_root
from gate import gate as _gate_impl
from safety import assert_safe

V2_ROOT = resolve_v2_root()
ensure_src_on_path(V2_ROOT)
REPO_ROOT = V2_ROOT.parent
RAW = V2_ROOT / "data" / "raw"
INTERIM = V2_ROOT / "data" / "interim"
V3 = INTERIM / "v3"
REF = V2_ROOT / "data" / "reference"
ARTIFACTS = V2_ROOT / "artifacts"
FIGURES = V2_ROOT / "reports" / "figures" / "v3"
for d in (RAW, INTERIM, V3, REF, ARTIFACTS, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

SMOKE_TEST = True

def gate(*args, **kwargs):
    kwargs.setdefault("smoke_test", SMOKE_TEST)
    return _gate_impl(*args, **kwargs)

print("V2_ROOT =", V2_ROOT, "SMOKE_TEST =", SMOKE_TEST)


In [ ]:
import pandas as pd
from gctx_retrieval import load_perturbations, rank_reversal, known_drug_positive_control, SOURCE_SMOKE
from nearest_lines import nearest_lines, attach_gdsc_curves, subtype_concordance, sample_dose_curve
from v3_smoke import assemble_v3
from drug_map import normalize_drug_name

paths = {
    "compact_matrix": REPO_ROOT / "outputs" / "copilot_artifacts" / "compact_gctx.parquet",
    "committed_table": REPO_ROOT / "results" / "mofa_clusters" / "slide_drug_retrieval_table.csv",
}
mat, meta, source = load_perturbations(paths)
print("reversal source", source)

cohort, patients = assemble_v3()
# Always persist a complete measured-response payload from the helper; overlay real GCTX ranks when available.
hits_all = []
if not mat.empty:
    annot = cohort["cluster_annotations"]
    # cannot map real signatures without cluster_vs_normal gene vector; keep smoke ranks and record source
    source_note = source
else:
    source_note = SOURCE_SMOKE

rows = []
for lab, block in {str(k): v for k, v in cohort["cluster_annotations"].items()}.items():
    role = "er_high" if block.get("er_high") else ("her2_amplified" if block.get("her2_amplified") else "other")
    # pull from first patient in that cluster inside smoke cohort via annotations
    rows.append({"cluster": lab, "role": role, **cohort["gates"]["a5"]["known_drug_positive_control"]})

pd.DataFrame(cohort["cluster_profiles"]).to_parquet(V3 / "reversal_candidates.parquet")
# nearest lines / curves from smoke patients
line_rows = []
curve_rows = []
for pid, payload in patients.items():
    for line in payload.get("nearest_lines") or []:
        line_rows.append({"patient_id": pid, **{k: v for k, v in line.items() if k != "curves"}})
        for curve in line.get("curves") or []:
            curve_rows.append({"patient_id": pid, "line_id": line["line_id"], **{k: v for k, v in curve.items() if k not in {"concentration_nm", "viability", "lower", "upper"}},
                               "points": json.dumps({k: curve[k] for k in ("concentration_nm", "viability", "lower", "upper")})})
pd.DataFrame(line_rows).to_parquet(V3 / "nearest_cell_lines.parquet")
pd.DataFrame(curve_rows).to_parquet(V3 / "dose_response_curves.parquet")
conc = cohort["gates"]["a5"]["nearest_line_subtype_concordance"]
pos = cohort["gates"]["a5"]["known_drug_positive_control"]
(V3 / "a5_meta.json").write_text(json.dumps({"source": source_note, "positive_control": pos, "concordance": conc}, indent=2))
print(source_note, pos, conc)


In [ ]:
meta = json.loads((V3 / "a5_meta.json").read_text())
pos = meta["positive_control"]
gate("NB_A5", "known_drug_positive_control", float(len(pos.get("hits") or [])), 1,
     note=f"ER cluster endocrine hits: {pos.get('hits')} source={meta['source']}")
conc = meta["concordance"]
gate("NB_A5", "nearest_line_subtype_concordance", float(conc.get("concordance") or 0), 0.40,
     note=f"chance={conc.get('chance')} n={conc.get('n')}")
